In [2]:
import json, re

In [15]:
text_file = '../data/raw_text/latin_grammar_junior_scholarship/Latin_Grammar_and_Junior_Scholarship_Pap.txt'
key_file = '../data/raw_text/latin_grammar_junior_scholarship/Key_to_Latin_grammar_and_junior_scholars.txt'

with open(text_file, 'r') as f:
    text = f.read()
with open(key_file, 'r') as f:
    key_ = f.read()

text_lines = text.split('\n')
key_lines = key_.split('\n')

In [1]:
text_lines

NameError: name 'text_lines' is not defined

In [74]:
def section_split(lines):
    curr_section = []
    curr_section_num = '0'
    all_sections = {}

    for line in lines:
        match_ = re.match('(\d+)\.$', line)
        appendix_match = re.match('APPENDIX ([IV]+)\.$', line)
        if match_: # starting new section
            curr_txt = '\n'.join(curr_section)
            all_sections[curr_section_num] = curr_txt
            curr_section = []

            curr_section_num = match_.group(1)
        elif appendix_match:
            curr_txt = '\n'.join(curr_section)
            all_sections[curr_section_num] = curr_txt
            curr_section = []

            curr_section_num = 'app.'+appendix_match.group(1)
        elif line == '':
            continue
        else:
            curr_section.append(line)

    # add last section
    curr_txt = '\n'.join(curr_section)
    all_sections[curr_section_num] = curr_txt 

    return all_sections

In [11]:
line = '1.'
match_ = re.match('(\d+)\.$', line)
match_

<re.Match object; span=(0, 2), match='1.'>

In [75]:
# split into sections
# sections will start with a line like: '1.'

key_sections = section_split(key_lines)
text_sections = section_split(text_lines)

In [76]:
key_sections

{'0': "This is a reproduction of a library book that was digitized\nby Google as part of an ongoing effort to preserve the\ninformation in books and make it universally accessible.\nGoogle books\nhttps://books.google.com\nKEY\nTO\nLATIN GRAMMAR\nAND\nJUNIOR SCHOLARSHIP\nPAPERS\nBY THE REV.\nJ. H. RAVEN, M.A.\nHEADMASTER OF THE FAUCONBERGE SCHOOL, BECCLES, SUFFOLK\nN.B.—This Key is for the use of Tutors only, and is issued\non the understanding that it shall not get into the hands of\nany Pupil.\nRIVINGTONS\nWATERLOO PLACE, LONDON\n[Price Five Shillings]\nENGLISH SCHOOL-CLASSICS\nEDITED BY FRANCIS STORR, M.A.,\nCHIEF MASTER OF MODERN SUBJECTS IN MERCHANT TAYLORS' SCHOOL\nTHOMSON'S SEASONS: Winter.\nWith an Introduction to the Series. By the Rev. J. F. BRIGHT.\nCOWPER'S TASK.\nBy FRANCIS STORR, M.A.\nPart I. (Book I.—The Sofa; Book II.—The Timepiece) \nPart II. (Book III.—The Garden; Book IV.—The Winter Evening) \nPart III. (Book V.—The Winter Morning Walk; Book VI.—The Winter Walk at No

In [77]:
text_sections

{'0': "This is a reproduction of a library book that was digitized\nby Google as part of an ongoing effort to preserve the\ninformation in books and make it universally accessible.\nGoogle™ books\nhttps://books.google.com\nLATIN GRAMMAR\nAND\nJUNIOR SCHOLARSHIP\nPAPERS\n_J. H. RAVEN_\nRIVINGTONS\nVINGTON'S MATHEMATICAL SERIES\nBy J. HAMBLIN SMITH, M.A.,\nGONVILLE AND CAIUS COLLEGE, AND LATE LECTURER AT ST. PETER'S\nCOLLEGE, CAMBRIDGE,\nArithmetic. 3s. 6d. A KEY, 9s.\nElementary Algebra. 3s. Without Answers, 2s. 6d.\nA KEY, 9s.\nExercises on Algebra. 2s. 6d.\n[Copies may be had without the Answers.]\nElementary Trigonometry. 4s. 6d. A KEY, 7s. 6d.\nElements of Geometry.\nContaining Books 1 to 6, and portions of Books 11 and 12 of\nEUCLID, with Exercises and Notes. 3s. 6d. A KEY, 8s. 6d.\nBooks 1 and 2, 1s. 6d., may be had separately.\nElementary Statics. 3s.  } A KEY, 6s.\nElementary Hydrostatics. 3s. }\nBook of Enunciations \nFOR HAMBLIN SMITH'S GEOMETRY, ALGEBRA, TRIGONO-\nMETRY, STAT

In [78]:
# for each section, split further into questions

def question_split(section_txt):
    
    questions = re.split('(\d+)\.', section_txt)
    # remove any empty lines
    new_questions = [q for q in questions if q != '' and q != '*' and q != '[']

    # find the first number
    idx = 0
    for i in range(len(new_questions)):
        if new_questions[i].isnumeric():
            idx = i
            break 

    new_questions = new_questions[idx:]
    print(new_questions)
    
    # make dict num -> txt
    dict_ = {}
    for i in range(0, len(new_questions), 2):
        num = new_questions[i]
        txt = new_questions[i+1]
        dict_[num] = txt.strip()

    return dict_

In [79]:
new_key_secs = {}
new_txt_secs = {}
skip_secs = ['0', '88', '104']
for sec_num in key_sections:
    # skip 0
    if sec_num in skip_secs: continue
    if sec_num not in text_sections: continue
    print('section:', sec_num)

    question_dict = question_split(text_sections[sec_num])
    key_dict = question_split(key_sections[sec_num])

    for q_num in key_dict:
        if q_num not in question_dict: continue

        new_key = f"{sec_num}.{q_num}"
        new_key_secs[new_key] = key_dict[q_num]
        new_txt_secs[new_key] = question_dict[q_num]

section: 1
['1', ' What nouns of decl. 1 and 2 make gen. plur. in -ûm?\n', '2', ' Decline—in sing.: Æneas, Cybele, respublica, deus, filius, pelagus, jusjurandum; in plur.: dea, Atrides, deus, faber, sestertius.\n', '3', ' Give the gender, meaning, and gen. sing. of—lis, fax, caput, auceps, accipiter, iter, supellex, grex, requies, simultas, mos, as, aes, livor, nix, os (2), frons (2), bidens (2), acus, dies, virus.\n', '4', ' What are the locatives of—humus, Roma, Athenae, bellum, Gades?\n', '5', ' Explain "heteroclite" and "heterogeneous;" and give two examples of nouns of each kind.\n', '6', ' Decline—domus; and give Latin for—"at home," "from home," "(to) home."\n', '7', ' Form patronymics, masc. and fem., from—Nereus, Thestius; and diminutives from—rivus, pars, homo, filius, flos, patera.\n', '8', ' Give the fem. of—Thrax, Tros, gallus, taurus, aries.\n', '9', ' Show by an example of each the force of the terminations -ile, -etum, -tor, -trix, -arium, -ulum.\n', '10', ' Distinguis

In [86]:
new_key_secs

{'1.1': 'Of decl. 1: comps. of _colo_ and _gigno_; _drachma_ and _amphora_; patronymics; names of some peoples.\nOf decl. 2: weights, coins, trades, measures; dissyllables of short penult (in poetry); names of peoples. N.B. Dist. numerals use this contraction.',
 '1.2': 'N. Æneas, Cybele, respublica, deus\nV. Æneas, Cybele, respublica, deus\nAc. Ænean, -am, Cybelen, rempublicam, deum\nG. Æneæ, Cybeles, -æ, reipublicæ, dei\nD. Æneæ, Cybelæ, reipublicae, deo\nAb. Ænea, Cybele, republicā, deo\nN. filius, pelagus, jusjurandum\nV. fili, pelagus, jusjurandum\nAc. filium, pelagus, jusjurandum\nG. filii, pelagi, jurisjurandi\nD. filio, pelago, jurijurando\nAb. filio, pelago, jurejurando\nN.V. deæ, Atrides, dei, dii, di, fabri, sestertii\nAc. deas, Atridas, deos, fabros, sestertios\nG. dearum, Atridum, deorum, deûm, fabrûm, sestertiûm\nD. deabus, Atridis, deis, diis, dis, fabris, sestertiis\nAb. deabus, Atridis, deis, diis, dis, fabris, sestertiis',
 '1.3': 'Fem., lawsuit, _litis_; fem., torch,

In [85]:
new_txt_secs

{'1.1': 'What nouns of decl. 1 and 2 make gen. plur. in -ûm?',
 '1.2': 'Decline—in sing.: Æneas, Cybele, respublica, deus, filius, pelagus, jusjurandum; in plur.: dea, Atrides, deus, faber, sestertius.',
 '1.3': 'Give the gender, meaning, and gen. sing. of—lis, fax, caput, auceps, accipiter, iter, supellex, grex, requies, simultas, mos, as, aes, livor, nix, os (2), frons (2), bidens (2), acus, dies, virus.',
 '2.1': 'What parisyllabic nouns of decl. 3 make gen. plur. in -*um* for -*ium*?',
 '2.2': 'Decline, giving all forms used—in sing.: *navis, requies, laurus, tussis*; in plur.: *jugerum, quercus, mensis, civitas*.',
 '2.3': 'Give the gender, meaning, and gen. sing. of—*jecur, plebs, plebes, femur, crambe, sidus, mas, tus*.',
 '2.4': 'Give the acc. sing. of—*aër, aether, Thisbe, Delos, Peleus, vis*.',
 '2.5': 'What other forms are used for—*praesepe, penus, tapetum, vespere, nocte*?',
 '2.6': 'Give instances of nouns not following the usual rules of flexion to avoid confusion with o

In [84]:
delete_keys = [
    'app.V.1485',
    'app.V.1688',
    'app.V.1837',
    'app.V.1882'
]

for key_ in delete_keys:
    if key_ in new_key_secs:
        del new_key_secs[key_]
    if key_ in new_txt_secs:
        del new_txt_secs[key_]

In [87]:
len(new_key_secs)

836

In [88]:
with open('../data/semi_structured/junior_scholarship.json', 'w') as f:
    json.dump(
        {
            'questions': new_txt_secs,
            'answers': new_key_secs
        },
        f,
        indent=4
    )

prompting to further split questions, disambiguate, filter, classify question types

In [3]:
with open('../data/semi_structured/junior_scholarship.json', 'r') as f:
    data = json.load(f)

new_txt_secs = data['questions']
new_key_secs = data['answers']


In [4]:
from openai import OpenAI
from time import sleep


with open('openai-api-key.txt', 'r') as f:
    openai_api_key = f.read().strip()

openai_client = OpenAI(api_key=openai_api_key)

In [5]:
PROMPT_TEMPLATE = '''I will give you a question and its answer.
First, decide if the question is answerable from the given context. If not, simply respond "SKIP". If it is easy to add context, then add the necessary context to the question. The question comes from a textbook on Latin grammar, so the question relates to Latin grammar, literature, or history.
Next, can the question be split into multiple questions? For example, if the question asks to list the meanings of multiple words, then it can be split into separate questions for each word.
Then, identify the language of the question and answer. You should only label the question language as "latin" if the question itself is asked in Latin, NOT if latin words or sentences are in the question.
Finally, identify the question content. These can be knowledge based (history, mythology, geography, literature) or skill based (reading comprehension, vocabulary, grammar, translation, literary devices, or scansion).

Organize your response in the following format:
Metadata:
question_language: [latin or english]
answer_language: [latin, english, or both]
question_content: [history, mythology, geography, literature, reading comprehension, vocabulary, grammar, translation, literary devices, or scansion]

Questions:
1. question text
Answer: answer text
2. question text
Answer: answer text
...

Here is the question and answer:
{}

{}

'''

In [7]:
n = '5.2'
q = new_txt_secs[n]
a = new_key_secs[n]

prompt =PROMPT_TEMPLATE.format(q, a)
print(prompt)

I will give you a question and its answer.
First, decide if the question is answerable from the given context. If not, simply respond "SKIP". If it is easy to add context, then add the necessary context to the question. The question comes from a textbook on Latin grammar, so the question relates to Latin grammar, literature, or history.
Next, can the question be split into multiple questions? For example, if the question asks to list the meanings of multiple words, then it can be split into separate questions for each word.
Then, identify the language of the question and answer. You should only label the question language as "latin" if the question itself is asked in Latin, NOT if latin words or sentences are in the question.
Finally, identify the question content. These can be knowledge based (history, mythology, geography, literature) or skill based (reading comprehension, vocabulary, grammar, translation, literary devices, or scansion).

Organize your response in the following forma

In [13]:
in_cost = 2.5 # per 1M
out_cost = 10 # per 1M 

In [8]:
response = openai_client.responses.create(
        model="gpt-4o",
        input=prompt
    )

In [14]:
print(response.output[0].content[0].text)

Metadata:
question_language: english
answer_language: english
question_content: grammar

Questions:
1. Explain "proper," as applied to nouns, with an example.
Answer: "Proper," relating to a particular person or place, as Caesar, Roma.
2. Explain "concrete," as applied to nouns, with an example.
Answer: "Concrete," relating to something actual and real, perceptible to the bodily senses, as *mons*, *panis*.
3. Explain "abstract," as applied to nouns, with an example.
Answer: "Abstract," to things ideal, imperceptible to the bodily senses, as *virtus*, *mendacitas*.
4. Explain "collective," as applied to nouns, with an example.
Answer: "Collective," to an aggregate of single separate objects, as *populus*.
5. Explain "epicene," as applied to nouns, with an example.
Answer: "Epicene," to nouns which can be either male or female, as *vulpes*.


In [15]:
q_words = len(prompt.split())
q_toks = q_words * 4 / 3
in_cost = q_toks * in_cost * len(new_txt_secs) / 1000000
in_cost

0.7830533333333334

In [11]:
resp_txt = response.output[0].content[0].text
resp_words = len(resp_txt.split())
resp_toks = resp_words * 4 / 3

In [16]:
out_cost = resp_toks * out_cost * len(new_txt_secs) / 1000000
out_cost

1.37104

In [17]:
model_responses = {}
N = len(new_txt_secs)
for i, q_id in enumerate(new_txt_secs): 

    q = new_txt_secs[q_id]
    a = new_key_secs[q_id]
    prompt = PROMPT_TEMPLATE.format(q, a)
    response = openai_client.responses.create(
        model="gpt-4o",
        input=prompt
    )

    try: 
        resp_txt = response.output[0].content[0].text
        model_responses[q_id] = resp_txt
    except Exception as e:
        print(f"Error processing question {q_id}: {e}")
        model_responses[q_id] = "Error"

    # every 50 questions, dump to file
    if i % 50 == 0:
        print(f"Dumping model responses to file {i} of {N}")
        with open(f'../data/semi_structured/junior_scholarship_model_responses.json', 'w') as f:
            json.dump(model_responses, f, indent=4)

    sleep(.5)


Dumping model responses to file 0 of 836
Dumping model responses to file 50 of 836
Dumping model responses to file 100 of 836
Dumping model responses to file 150 of 836
Dumping model responses to file 200 of 836
Dumping model responses to file 250 of 836
Dumping model responses to file 300 of 836
Dumping model responses to file 350 of 836
Dumping model responses to file 400 of 836
Dumping model responses to file 450 of 836
Dumping model responses to file 500 of 836
Dumping model responses to file 550 of 836
Dumping model responses to file 600 of 836
Dumping model responses to file 650 of 836
Dumping model responses to file 700 of 836
Dumping model responses to file 750 of 836
Dumping model responses to file 800 of 836


In [18]:
# dump final model responses
with open(f'../data/semi_structured/junior_scholarship_model_responses.json', 'w') as f:
    json.dump(model_responses, f, indent=4)

parsing model responses


In [3]:
with open('../data/semi_structured/junior_scholarship_model_responses.json', 'r') as f:
    model_responses = json.load(f)

In [4]:
skip_secs = ['18']

In [22]:
def parse_response(response: str, q_id_prefix: str, metadata: dict):
    lines = response.split('\n')

    question_language = ''
    lang_match = re.search(r'question_language: (\w+)\n', response)
    if lang_match:
        question_language = lang_match.group(1)
    else:
        print(f"No question language found for {q_id_prefix}")

    answer_language = ''
    lang_match = re.search(r'answer_language: (\w+)\n', response)
    if lang_match:
        answer_language = lang_match.group(1)
    else:
        print(f"No answer language found for {q_id_prefix}")

    question_content = ''
    content_match = re.search(r'question_content: ([\w\s,]+)\n', response)
    if content_match:
        question_content = content_match.group(1).strip()
    else:
        print(f"No question content found for {q_id_prefix}")

    # parse questions
    # find line with "Questions:"
    idx = 0
    for i, line in enumerate(lines):
        if re.match(r'^(\**)Question(s?)(\**):', line):
            idx = i
            break

    questions = lines[idx+1:]


    # parse questions
    parsed_questions = []
    i = 0
    curr_num = ''
    curr_q_lines = []
    curr_a_lines = []
    doing_q = False
    doing_a = False
    for i in range(len(questions)):
        line = questions[i]
        
        # check if line is a question: it starts with a number
        q_match = re.match(r'^(\d+)\.', line)
        a_match = re.match(r'^\s*Answer:\s*', line)
        if q_match:

            # add curr question to parsed questions
            if curr_num:
                q_text = '\n'.join(curr_q_lines).strip()
                a_text = '\n'.join(curr_a_lines).strip()

                this_metadata = metadata.copy()
                this_metadata['question_text'] = q_text
                this_metadata['answers'] = [a_text]
                this_metadata['question_id'] = q_id_prefix + curr_num
                this_metadata['question_content'] = question_content
                this_metadata['question_language'] = question_language
                this_metadata['answer_language'] = answer_language

                parsed_questions.append(this_metadata)


            # now we're starting on new question
            curr_num = q_match.group(1)
            curr_q_lines = []
            curr_a_lines = []
            doing_q = True
            doing_a = False
                
            # strip number and period
            q_text = line[len(curr_num)+1:].strip()

            curr_q_lines.append(q_text)

        elif a_match:
            # strip "Answer:" from start
            txt = line[len(a_match.group(0)):]
            curr_a_lines.append(txt)
            doing_q = False 
            doing_a = True
        
        else:
            if doing_q:
                curr_q_lines.append(line)
            elif doing_a:
                curr_a_lines.append(line)

    # add last question/answer
    if curr_a_lines and curr_q_lines:
        q_text = '\n'.join(curr_q_lines).strip()
        a_text = '\n'.join(curr_a_lines).strip()

        this_metadata = metadata.copy()
        this_metadata['question_text'] = q_text
        this_metadata['answers'] = [a_text]
        this_metadata['question_id'] = q_id_prefix + curr_num
        this_metadata['question_content'] = question_content
        this_metadata['question_language'] = question_language
        this_metadata['answer_language'] = answer_language

        parsed_questions.append(this_metadata)
        
    return parsed_questions

In [23]:
source = 'latin-grammar-junior-scholarship'
year = 1884
difficulty = 'unknown'
format = 'short_answer'

all_questions = []

for sec_num in model_responses:
    if any(sec_num.startswith(skip_sec) for skip_sec in skip_secs): continue
    response = model_responses[sec_num]
    if response == 'SKIP' or 'SKIP' in response: continue
    if response == 'Error': continue
    if response == '': continue

    # construct known metadata
    q_id_prefix = source + '_' + sec_num + '.'

    metadata = {
        'source_name': source,
        'source_year': year,
        'question_id': q_id_prefix,
        'question_format': format,
        'question_content': '', # filled in parse_response,
        'difficulty': difficulty,
        'question_language': '', # filled in parse_response,
        'answer_language': '', # filled in parse_response,
        'question_text': '', # filled in parse_response,
        'multiple_choice_options': [], # empty for short answer
        'answers': [] # filled in parse_response
    }

    # parse response
    new_questions = parse_response(response, q_id_prefix, metadata)
    all_questions.extend(new_questions)



        

No question content found for latin-grammar-junior-scholarship_20.4.
No answer language found for latin-grammar-junior-scholarship_27.1.
No question language found for latin-grammar-junior-scholarship_55.7.
No answer language found for latin-grammar-junior-scholarship_55.7.
No question language found for latin-grammar-junior-scholarship_91.10.
No answer language found for latin-grammar-junior-scholarship_91.10.
No question language found for latin-grammar-junior-scholarship_93.2.
No answer language found for latin-grammar-junior-scholarship_93.2.
No answer language found for latin-grammar-junior-scholarship_96.8.
No question content found for latin-grammar-junior-scholarship_app.V.6.


In [24]:
all_questions

[{'source_name': 'latin-grammar-junior-scholarship',
  'source_year': 1884,
  'question_id': 'latin-grammar-junior-scholarship_1.1.1',
  'question_format': 'short_answer',
  'question_content': 'grammar',
  'difficulty': 'unknown',
  'question_language': 'english',
  'answer_language': 'english',
  'question_text': 'Which nouns of the first declension make the genitive plural in -ûm?',
  'multiple_choice_options': [],
  'answers': ['Compounds of _colo_ and _gigno_; _drachma_ and _amphora_; patronymics; names of some peoples.']},
 {'source_name': 'latin-grammar-junior-scholarship',
  'source_year': 1884,
  'question_id': 'latin-grammar-junior-scholarship_1.1.2',
  'question_format': 'short_answer',
  'question_content': 'grammar',
  'difficulty': 'unknown',
  'question_language': 'english',
  'answer_language': 'english',
  'question_text': 'Which nouns of the second declension make the genitive plural in -ûm?',
  'multiple_choice_options': [],
  'answers': ['Weights, coins, trades, mea

In [25]:
len(all_questions)

3746

In [26]:
with open('../data/structured/junior_scholarship.json', 'w') as f:
    json.dump(all_questions, f, indent=4)